In [1]:
# imports
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


In [2]:
# load the data
data = pd.read_csv('data.csv')
data.head()

,statename,year,month,spend apparel and accessories,spend accommodation and food services,"spend arts, entertainment, and recreation",spend all,spend general merchandise + apparel,spend durable goods,spend general merchandise,...,hospitalized_rate,day,time retail and recreation,time grocery and pharmacy,time parks,time transit stations,time workplaces,time residential,time away from home,emp recovered
0,Alabama,2020,1,0.0000,-1.910000e-09,0.0000,0.0000,-2.880000e-08,3.280000e-08,0.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,Alabama,2020,2,-0.0438,2.160000e-02,0.0643,0.0278,2.020000e-02,1.250000e-02,0.0782,...,NaN,29.0,0.0900,0.0414,0.191,0.1070,0.010,-0.00571,0.00855,True
2,Alabama,2020,3,-0.3510,-3.800000e-01,-0.3730,-0.0898,-1.460000e-01,-3.850000e-02,0.0646,...,0.101591,31.0,-0.3340,-0.0643,0.180,-0.2340,-0.344,0.13100,-0.15900,False
3,Alabama,2020,4,-0.5350,-6.060000e-01,-0.6520,-0.2290,-3.190000e-01,1.530000e-02,-0.0916,...,7.704000,30.0,-0.2400,-0.0300,0.143,-0.2300,-0.347,0.12100,-0.14800,False
4,Alabama,2020,5,-0.1820,-3.820000e-01,-0.4720,-0.1010,-7.270000e-02,1.720000e-01,0.0279,...,8.569677,31.0,-0.0914,0.0314,0.373,-0.0371,-0.236,0.08290,-0.09430,False


In [3]:
data.columns

Index(['statename', 'year', 'month', 'spend apparel and accessories',
       'spend accommodation and food services',
       'spend arts, entertainment, and recreation', 'spend all',
       'spend general merchandise + apparel', 'spend durable goods',
       'spend general merchandise', 'spend grocery and food stores',
       'spend health care and social assistance',
       'spend home improvement centers', 'spend other in-person services',
       'spend non-durable goods', 'spend remote services',
       'spend transportation and warehousing', 'spend all_incmiddle',
       'spend all q1', 'spend all q2', 'spend all q3', 'spend all q4',
       'spend in-person services', 'spend retail excluding grocery',
       'spend retail including grocery', 'day_endofweek', 'emp all', 'emp q1',
       'emp q2', 'emp q3', 'emp q4', 'emp inc middle', 'emp inc below_median',
       'emp inc above median', 'emp trade_transport utilities',
       'emp professional business_services', 'emp education hea

### 1. data prep

In [12]:
# select features and target
feat = ['spend apparel and accessories',  'spend grocery and food stores', # spend features
        'case_rate', 'death_rate', 'hospitalized_rate', # covid features, 
        ]
target = 'emp all'

# see missing values in features and target
print(f'num missing in cols: \n{data[feat + [target]].isnull().sum()}')

# drop rows with missing values
data = data.dropna(subset=feat + [target])

# store statenames for mapping later
state_names = data['statename']

# define X and y
X = data[feat]
y = data[target]

X.head()


num missing in cols: 
spend apparel and accessories    0
spend grocery and food stores    0
case_rate                        0
death_rate                       0
hospitalized_rate                0
emp all                          0
dtype: int64


,spend apparel and accessories,spend grocery and food stores,case_rate,death_rate,hospitalized_rate
2,-0.3510,0.3740,3.848895,0.017158,0.101591
3,-0.5350,0.1650,72.970000,2.257533,7.704000
4,-0.1820,0.1500,224.387097,8.786129,8.569677
5,-0.0250,0.0782,504.966667,15.463333,16.526667
6,-0.0983,0.1130,1166.548387,24.080645,32.906452


### 2. define train and test sets

In [13]:
# test set is all data from january 2025 (five years after the start of the pandemic)
test_mask = (data['year'] == 2025) & (data['month'] == 1)
X_test = X[test_mask]
y_test = y[test_mask]
states_test = state_names[test_mask]

# train set is all other data
X_train = X[~test_mask]
y_train = y[~test_mask]
states_train = state_names[~test_mask]

# print lengths of train and test sets
print(f'train set length: {len(X_train)}')
print(f'test set length: {len(X_test)}')

train set length: 3133
test set length: 50


### 3. initialize and fit the model

In [14]:
# initilize the model
lin_reg = LinearRegression()

# fit the model
lin_reg.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


### 4. make predictions

In [15]:
# use fitted model to make predictions
y_pred = lin_reg.predict(X_test)

### 5. evaluate the model

In [17]:
import statsmodels.api as sm

# Add constant (intercept)
X = sm.add_constant(X)

# Build model
model = sm.OLS(y, X).fit()

# View summary
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                emp all   R-squared:                       0.158
Model:                            OLS   Adj. R-squared:                  0.157
Method:                 Least Squares   F-statistic:                     119.2
Date:                Mon, 03 Nov 2025   Prob (F-statistic):          6.91e-116
Time:                        09:07:39   Log-Likelihood:                 2806.5
No. Observations:                3183   AIC:                            -5601.
Df Residuals:                    3177   BIC:                            -5565.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
const         